# TorchScript DINO Validation

This notebook compares the later hybrid Python mask logic from the DINO reference workflow against the frozen offline C++ replay artifacts produced from the single-channel TorchScript validation path.

Expected sequence:
1. Capture one tensor and one live DINO mask with `run_torchscript_validation_capture_single_channel.sh`.
2. Run `run_offline_dino_validator.sh` on that tensor snapshot.
3. Run this notebook to compare the Python later-hybrid mask against the offline and live C++ masks.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage

WORKSPACE_ROOT = Path('/home/sat3737/holoscan_demo_workspace')
REFERENCE_REPO = WORKSPACE_ROOT / 'Dinov3-RF-Signal-Detection'
HELPERS_PATH = REFERENCE_REPO / 'signal_detection_holoscan_retry_dino_helpers.py'

helpers_spec = importlib.util.spec_from_file_location('signal_detection_holoscan_retry_dino_helpers', HELPERS_PATH)
if helpers_spec is None or helpers_spec.loader is None:
    raise RuntimeError(f'Unable to load helper module from {HELPERS_PATH}')
dino_helpers = importlib.util.module_from_spec(helpers_spec)
helpers_spec.loader.exec_module(dino_helpers)

run_subsection_dino_texture_experiment = dino_helpers.run_subsection_dino_texture_experiment

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['image.cmap'] = 'viridis'


In [ ]:
def load_yaml_scalar(text: str, key: str, cast=float, default=None):
    match = re.search(rf'(^|\n)\s*{re.escape(key)}\s*:\s*("[^"]*"|[^\n]+)', text)
    if not match:
        return default
    raw = match.group(2).strip()
    if raw.startswith('\"') and raw.endswith('\"'):
        raw = raw[1:-1]
    if cast is str:
        return raw
    return cast(raw)


def load_validation_config(path: Path) -> dict[str, object]:
    text = path.read_text()
    return {
        'model_name': load_yaml_scalar(text, 'model_name', str, 'dinov3_vitb16'),
        'model_repo_path': load_yaml_scalar(text, 'model_repo_path', str, str(WORKSPACE_ROOT / 'dinov3')),
        'weights_path': load_yaml_scalar(text, 'weights_path', str, ''),
    }


def read_pgm_mask(path: Path) -> np.ndarray:
    with path.open('rb') as handle:
        magic = handle.readline().strip()
        if magic != b'P5':
            raise ValueError(f'Unsupported PGM magic in {path}: {magic!r}')
        header_tokens = []
        while len(header_tokens) < 3:
            line = handle.readline()
            if not line:
                raise ValueError(f'Truncated PGM header in {path}')
            line = line.strip()
            if not line or line.startswith(b'#'):
                continue
            header_tokens.extend(line.split())
        cols, rows, max_value = map(int, header_tokens[:3])
        if max_value <= 0 or max_value > 255:
            raise ValueError(f'Unsupported PGM max value {max_value} in {path}')
        payload = handle.read(rows * cols)
        if len(payload) != rows * cols:
            raise ValueError(f'Truncated PGM payload in {path}')
    return np.frombuffer(payload, dtype=np.uint8).reshape(rows, cols) >= 128


def resize_valid_row_mask(source_rows: int, output_rows: int, output_cols: int, ignore_bins_per_side: int) -> np.ndarray:
    mask = np.zeros((output_rows, output_cols), dtype=bool)
    for output_row in range(output_rows):
        source_row = min(source_rows - 1, int(output_row * source_rows / max(output_rows, 1)))
        if ignore_bins_per_side <= source_row < (source_rows - ignore_bins_per_side):
            mask[output_row, :] = True
    return mask


def normalize01_masked(array: np.ndarray, mask: np.ndarray) -> np.ndarray:
    array = np.asarray(array, dtype=np.float32)
    active = np.asarray(mask, dtype=bool) & np.isfinite(array)
    if not np.any(active):
        return np.zeros_like(array, dtype=np.float32)
    low = float(np.min(array[active]))
    high = float(np.max(array[active]))
    if high <= low + 1.0e-12:
        return np.zeros_like(array, dtype=np.float32)
    return np.clip((array - low) / (high - low), 0.0, 1.0).astype(np.float32)


def keep_large_components(mask: np.ndarray, min_size: int) -> np.ndarray:
    labels, count = ndimage.label(np.asarray(mask, dtype=bool))
    if count <= 0:
        return np.zeros_like(mask, dtype=bool)
    sizes = ndimage.sum(np.ones_like(labels, dtype=np.uint8), labels, index=np.arange(1, count + 1))
    keep = np.where(np.asarray(sizes) >= float(min_size))[0] + 1
    if keep.size == 0:
        return np.zeros_like(mask, dtype=bool)
    return np.isin(labels, keep)


def mask_metrics(lhs: np.ndarray, rhs: np.ndarray) -> dict[str, float]:
    lhs = np.asarray(lhs, dtype=bool)
    rhs = np.asarray(rhs, dtype=bool)
    if lhs.shape != rhs.shape:
        raise ValueError(f'Mask shape mismatch: {lhs.shape} vs {rhs.shape}')
    agreement = float(np.mean(lhs == rhs))
    intersection = float(np.count_nonzero(lhs & rhs))
    union = float(np.count_nonzero(lhs | rhs))
    return {
        'agreement': agreement,
        'iou': 1.0 if union == 0.0 else intersection / union,
        'lhs_fraction': float(np.mean(lhs)),
        'rhs_fraction': float(np.mean(rhs)),
    }


def build_python_later_hybrid_mask(dino_score_px: np.ndarray, coherence_gate_px: np.ndarray, valid_mask: np.ndarray) -> dict[str, np.ndarray | float]:
    base_map = np.asarray(dino_score_px, dtype=np.float32) * np.asarray(coherence_gate_px, dtype=np.float32)
    base_norm = normalize01_masked(base_map, valid_mask)
    envelope_map = normalize01_masked(ndimage.gaussian_filter(base_norm, sigma=(6.0, 1.4)), valid_mask)
    residual_penalty = normalize01_masked(ndimage.gaussian_filter(np.abs(base_norm - ndimage.gaussian_filter(base_norm, sigma=(4.0, 1.0))), sigma=(2.0, 0.8)), valid_mask)
    freq_curvature_penalty = normalize01_masked(np.abs(ndimage.gaussian_filter1d(base_norm, sigma=0.8, axis=0, order=2)), valid_mask)
    keep_freq = normalize01_masked(envelope_map - 0.90 * freq_curvature_penalty, valid_mask)
    keep_res = normalize01_masked(envelope_map - 1.00 * residual_penalty, valid_mask)
    residual_veto_gate = np.clip((keep_res - 0.30) / 0.70, 0.0, 1.0).astype(np.float32)
    combined_score = normalize01_masked(keep_freq * (0.35 + 0.65 * residual_veto_gate), valid_mask)
    active_freq = keep_freq[valid_mask]
    active_res = keep_res[valid_mask]
    active_combined = combined_score[valid_mask]
    seed_freq_thr = float(np.quantile(active_freq, 0.90)) if active_freq.size else 1.0
    seed_res_thr = float(np.quantile(active_res, 0.82)) if active_res.size else 1.0
    combined_thr = float(np.quantile(active_combined, 0.78)) if active_combined.size else 1.0
    seed_mask = np.logical_and.reduce((keep_freq >= seed_freq_thr, keep_res >= seed_res_thr, valid_mask))
    seed_mask = keep_large_components(seed_mask, min_size=8)
    final_mask = np.logical_and(seed_mask, combined_score >= combined_thr * 0.85)
    final_mask = ndimage.binary_closing(final_mask, structure=np.ones((7, 3), dtype=bool), iterations=1)
    final_mask = ndimage.binary_fill_holes(final_mask)
    final_mask = keep_large_components(final_mask, min_size=24)
    final_mask = np.logical_and(final_mask, valid_mask)
    return {
        'hybrid_dino_contrib': base_map.astype(np.float32),
        'base_norm': base_norm,
        'envelope_map': envelope_map,
        'residual_penalty': residual_penalty,
        'freq_curvature_penalty': freq_curvature_penalty,
        'keep_freq': keep_freq,
        'keep_res': keep_res,
        'residual_veto_gate': residual_veto_gate,
        'combined_score': combined_score,
        'seed_mask': seed_mask,
        'final_mask': final_mask,
        'seed_freq_threshold': seed_freq_thr,
        'seed_res_threshold': seed_res_thr,
        'combined_threshold': combined_thr,
    }


In [ ]:
CONFIG_PATH = WORKSPACE_ROOT / 'holohub-dev' / 'applications' / 'usrp_wideband_signal_detection' / 'config_torchscript_validation_capture_single_channel.yaml'
ARTIFACT_ROOT = Path('/tmp/usrp_spectrograms/dino_validator_artifacts')
TENSOR_ROOT = Path('/tmp/usrp_spectrograms/tensors')
LIVE_MASK_ROOT = Path('/tmp/usrp_dino_masks')

summary_candidates = sorted(ARTIFACT_ROOT.glob('**/offline_validation_summary.json'))
if not summary_candidates:
    raise RuntimeError('No offline_validation_summary.json found under /tmp/usrp_spectrograms/dino_validator_artifacts. Run the offline validator first.')

SUMMARY_PATH = summary_candidates[-1]
summary = json.loads(SUMMARY_PATH.read_text())
config = load_validation_config(CONFIG_PATH)

tensor_path = Path(summary['tensor_path'])
if not tensor_path.exists():
    tensor_path = TENSOR_ROOT / tensor_path.name

offline_corrected_resized_path = Path(summary['corrected_resized_npy'])
offline_dino_score_path = Path(summary['dino_score_npy'])
offline_coherence_gate_path = Path(summary['coherence_gate_npy'])
offline_final_mask_path = Path(summary['final_mask_npy'])
offline_final_mask_pgm = Path(summary['final_mask_pgm'])

live_mask_path = Path(summary.get('live_mask_path', '')) if summary.get('live_mask_path') else None
if live_mask_path is not None and not live_mask_path.exists():
    candidate = LIVE_MASK_ROOT / live_mask_path.name
    live_mask_path = candidate if candidate.exists() else live_mask_path

print({
    'summary': str(SUMMARY_PATH),
    'tensor': str(tensor_path),
    'offline_corrected_resized': str(offline_corrected_resized_path),
    'offline_dino_score': str(offline_dino_score_path),
    'offline_coherence_gate': str(offline_coherence_gate_path),
    'offline_final_mask': str(offline_final_mask_pgm),
    'live_mask': None if live_mask_path is None else str(live_mask_path),
})


In [ ]:
corrected_resized = np.load(offline_corrected_resized_path, allow_pickle=False).astype(np.float32)
offline_dino_score = np.load(offline_dino_score_path, allow_pickle=False).astype(np.float32)
offline_coherence_gate = np.load(offline_coherence_gate_path, allow_pickle=False).astype(np.float32)
offline_final_mask = np.load(offline_final_mask_path, allow_pickle=False) >= 0.5

source_rows = int(summary['canonical_rows'])
output_rows = int(summary['output_rows'])
output_cols = int(summary['output_cols'])
ignore_bins_per_side = int(summary['ignore_bins_per_side'])
valid_score_mask = resize_valid_row_mask(source_rows, output_rows, output_cols, ignore_bins_per_side)

valid_values = corrected_resized[valid_score_mask] if np.any(valid_score_mask) else corrected_resized.reshape(-1)
python_reference = run_subsection_dino_texture_experiment(
    corrected_resized,
    dino_repo_dir=config['model_repo_path'],
    dino_weights_path=config['weights_path'],
    dino_model_name=config['model_name'],
    dino_device='cuda',
    min_component_size=6,
    dino_feature_knn=8,
    dino_spatial_weight=0.35,
    dino_score_q=0.60,
    texture_knn=6,
    texture_q=0.90,
    dino_db_min=float(np.percentile(valid_values, 2.0)),
    dino_db_max=float(np.percentile(valid_values, 98.0)),
)

python_dino_score = normalize01_masked(np.asarray(python_reference['dino_score_px'], dtype=np.float32), valid_score_mask)
offline_dino_score_norm = normalize01_masked(offline_dino_score, valid_score_mask)
python_hybrid = build_python_later_hybrid_mask(python_dino_score, offline_coherence_gate, valid_score_mask)

live_mask = read_pgm_mask(live_mask_path) if live_mask_path is not None and live_mask_path.exists() else None

comparison = {
    'python_vs_offline': mask_metrics(python_hybrid['final_mask'], offline_final_mask),
    'python_dino_score_mae': float(np.mean(np.abs(python_dino_score[valid_score_mask] - offline_dino_score_norm[valid_score_mask]))) if np.any(valid_score_mask) else 0.0,
    'python_dino_score_corr': float(np.corrcoef(python_dino_score[valid_score_mask], offline_dino_score_norm[valid_score_mask])[0, 1]) if np.count_nonzero(valid_score_mask) > 4 else 1.0,
    'seed_freq_threshold': float(python_hybrid['seed_freq_threshold']),
    'seed_res_threshold': float(python_hybrid['seed_res_threshold']),
    'combined_threshold': float(python_hybrid['combined_threshold']),
}
if live_mask is not None:
    comparison['python_vs_live'] = mask_metrics(python_hybrid['final_mask'], live_mask)
    comparison['offline_vs_live'] = mask_metrics(offline_final_mask, live_mask)

comparison


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10), constrained_layout=True)
panels = [
    ('Corrected resized spectrogram', corrected_resized, 'magma'),
    ('Offline coherence gate', offline_coherence_gate, 'viridis'),
    ('Python DINO score', python_dino_score, 'inferno'),
    ('Python later hybrid mask', python_hybrid['final_mask'].astype(np.float32), 'gray'),
    ('Offline C++ final mask', offline_final_mask.astype(np.float32), 'gray'),
    ('Valid score mask', valid_score_mask.astype(np.float32), 'gray'),
]

for axis, (title, image, cmap) in zip(axes.flat, panels):
    rendered = axis.imshow(image.T, origin='lower', aspect='auto', cmap=cmap)
    axis.set_title(title, fontsize=10)
    axis.set_xticks([])
    axis.set_yticks([])
    fig.colorbar(rendered, ax=axis, fraction=0.046, pad=0.04)

plt.show()

overlay_fig, overlay_axes = plt.subplots(1, 3 if live_mask is not None else 2, figsize=(16, 5), constrained_layout=True)
if not isinstance(overlay_axes, np.ndarray):
    overlay_axes = np.array([overlay_axes])

overlay_axes[0].imshow(corrected_resized.T, origin='lower', aspect='auto', cmap='magma')
overlay_axes[0].imshow(np.ma.masked_where(~python_hybrid['final_mask'].T, python_hybrid['final_mask'].T), origin='lower', aspect='auto', cmap='cool', alpha=0.45)
overlay_axes[0].set_title('Python later hybrid overlay')
overlay_axes[0].set_xticks([])
overlay_axes[0].set_yticks([])

overlay_axes[1].imshow(corrected_resized.T, origin='lower', aspect='auto', cmap='magma')
overlay_axes[1].imshow(np.ma.masked_where(~offline_final_mask.T, offline_final_mask.T), origin='lower', aspect='auto', cmap='spring', alpha=0.45)
overlay_axes[1].set_title('Offline C++ overlay')
overlay_axes[1].set_xticks([])
overlay_axes[1].set_yticks([])

if live_mask is not None:
    overlay_axes[2].imshow(corrected_resized.T, origin='lower', aspect='auto', cmap='magma')
    overlay_axes[2].imshow(np.ma.masked_where(~live_mask.T, live_mask.T), origin='lower', aspect='auto', cmap='winter', alpha=0.45)
    overlay_axes[2].set_title('Live C++ overlay')
    overlay_axes[2].set_xticks([])
    overlay_axes[2].set_yticks([])

plt.show()
